In [ ]:
import os
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder, boxscoretraditionalv2
from requests.exceptions import ReadTimeout, ConnectionError

# 定义计算 Possessions 的函数
def calculate_possessions(fga, fta, orb, tov, opp_fga, opp_fta, opp_orb, opp_tov):
    possessions = 0.5 * ((fga + 0.4 * fta - orb + tov) + (opp_fga + 0.4 * opp_fta - opp_orb + opp_tov))
    return possessions

# 获取 2023-24 赛季的所有比赛信息
def get_all_games(season='2023-24'):
    print("Fetching all games...")
    gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=season)
    games = gamefinder.get_data_frames()[0]
    print(f"Total games found: {len(games)}")
    return games

# 获取单场比赛的统计数据并计算回合数
def get_game_stats_and_possessions(games, output_file='game_data.csv'):
    # 检查是否有已保存数据
    if os.path.exists(output_file):
        saved_data = pd.read_csv(output_file)
        saved_game_ids = set(saved_data['GAME_ID'])
        print(f"{len(saved_game_ids)} games already saved. Skipping these games...")
    else:
        saved_data = pd.DataFrame()
        saved_game_ids = set()

    all_game_data = []
    for _, row in games.iterrows():
        game_id = row['GAME_ID']

        if game_id in saved_game_ids:
            continue  # 跳过已保存的比赛

        try:
            for attempt in range(3):  # 最多尝试 3 次
                try:
                    time.sleep(1)  # 延迟以避免限流
                    boxscore = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id, timeout=60)
                    break  # 请求成功，退出重试循环
                except (ReadTimeout, ConnectionError) as e:
                    print(f"Error fetching game {game_id}: {e}. Retrying... (Attempt {attempt + 1})")
                    time.sleep(5)  # 失败后等待 5 秒重试
            else:
                print(f"Failed to fetch data for game {game_id} after 3 attempts.")
                continue

            game_data = boxscore.get_data_frames()[1]  # 获取 Team Stats 数据
            team_stats = game_data[['TEAM_ID', 'FGA', 'FTA', 'OREB', 'TO', 'PTS']]

            # 分成主客场两支球队数据
            team1 = team_stats.iloc[0]
            team2 = team_stats.iloc[1]

            # 计算 Possessions
            possessions = calculate_possessions(
                team1['FGA'], team1['FTA'], team1['OREB'], team1['TO'],
                team2['FGA'], team2['FTA'], team2['OREB'], team2['TO']
            )

            # 添加 Possessions 列到比赛信息
            row_data = row.to_dict()
            row_data['POSSESSIONS'] = possessions
            all_game_data.append(row_data)

            # 将数据写入 CSV（追加模式）
            pd.DataFrame([row_data]).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

        except Exception as e:
            print(f"Error processing game {game_id}: {e}")

    return pd.DataFrame(all_game_data)

# 批量处理比赛，避免请求过载
def process_games_in_batches(games, batch_size=100, output_file='game_data.csv'):
    total_games = len(games)
    all_game_data = []
    for start_idx in range(0, total_games, batch_size):
        end_idx = min(start_idx + batch_size, total_games)
        batch_games = games.iloc[start_idx:end_idx]
        print(f"Processing games {start_idx + 1} to {end_idx}...")
        batch_data = get_game_stats_and_possessions(batch_games, output_file=output_file)
        all_game_data.append(batch_data)
        time.sleep(10)  # 每处理一批等待 10 秒，防止限流
    return pd.concat(all_game_data, ignore_index=True)

# 主函数执行流程
if __name__ == "__main__":
    output_file = 'game_data.csv'  # 保存的本地文件
    games = get_all_games()  # 获取所有比赛
    possession_data = process_games_in_batches(games, output_file=output_file)
    print(f"Data saved to {output_file}")


Fetching all games...
Total games found: 4383
Processing games 1 to 100...
Processing games 101 to 200...
50 games already saved. Skipping these games...
Processing games 201 to 300...
104 games already saved. Skipping these games...
Processing games 301 to 400...
152 games already saved. Skipping these games...
